In [1]:

#import sys
#import ee
#import geemap

#print(sys.executable)
#print("ee e geemap importados")
#ee.Authenticate()   # abre o navegador para login, só na primeira vez
#ee.Initialize(project='projeto-estrela-513916')#

In [2]:
import ee, geemap
ee.Initialize(project='projeto-estrela-513916')
print('Conectado!')

Conectado!


In [3]:
area = ee.Geometry.Point([-51.96, -29.50]).buffer(15000)

s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(area)
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .select('VV'))

antes = s1.filterDate('2024-03-01', '2024-04-20').median().clip(area)
durante = s1.filterDate('2024-05-01', '2024-05-10').min().clip(area)
agua = durante.lt(-18).selfMask()

print('Imagens durante a cheia:', s1.filterDate('2024-05-01', '2024-05-10').size().getInfo())

Map = geemap.Map()
Map.centerObject(area, 11)
Map.addLayer(antes, {'min': -25, 'max': 0}, 'Antes da cheia (VV)')
Map.addLayer(durante, {'min': -25, 'max': 0}, 'Durante a cheia (VV)')
Map.addLayer(agua, {'palette': ['0000FF']}, 'Água provável')
Map

Imagens durante a cheia: 1


Map(center=[-29.499984611721658, -51.95999950969775], controls=(WidgetControl(options=['position', 'transparen…

In [4]:
import os
pasta = os.path.join(os.path.expanduser('~'), 'Downloads', 'sar_estrela')
os.makedirs(pasta, exist_ok=True)

# Mancha de água: 1 = água, 0 = seco
agua_export = durante.lt(-18).unmask(0).toByte().clip(area)
geemap.ee_export_image(
    agua_export,
    filename=os.path.join(pasta, 'agua_sentinel1_maio2024.tif'),
    scale=10,
    region=area,
    crs='EPSG:31982',
    file_per_band=False
)

# Imagem de radar durante a cheia (VV em dB)
geemap.ee_export_image(
    durante,
    filename=os.path.join(pasta, 'vv_durante_cheia.tif'),
    scale=20,
    region=area,
    crs='EPSG:31982',
    file_per_band=False
)

print('Arquivos salvos em:', pasta)

Generating URL ...
Please wait ...
Data downloaded to C:\Users\letic\Downloads\sar_estrela\agua_sentinel1_maio2024.tif
Generating URL ...
Please wait ...
Data downloaded to C:\Users\letic\Downloads\sar_estrela\vv_durante_cheia.tif
Arquivos salvos em: C:\Users\letic\Downloads\sar_estrela


In [5]:
# Entrada do porto de Rio Grande-RS (molhes e canal de acesso)
area = ee.Geometry.Point([-52.05, -32.13]).buffer(20000)

s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(area)
      .filterDate('2025-06-01', '2026-11-01')
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .sort('system:time_start', False))

imagem = s1.first().select('VV').clip(area)
data = ee.Date(s1.first().get('system:time_start')).format('YYYY-MM-dd HH:mm').getInfo()
print('Imagem de:', data)

# Máscara de água: pixels que têm água em mais de 50% do tempo (dados JRC)
agua = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').gt(50)

# Navios: pixels muito brilhantes sobre a água
navios = imagem.gt(-8).And(agua).selfMask()

Map = geemap.Map()
Map.centerObject(area, 11)
Map.addLayer(imagem, {'min': -25, 'max': 0}, f'Sentinel-1 VV ({data})')
Map.addLayer(navios, {'palette': ['FF0000']}, 'Possíveis navios')
Map

Imagem de: 2026-10-29 08:48


Map(center=[-32.12997948235455, -52.04999932811431], controls=(WidgetControl(options=['position', 'transparent…

In [6]:
# Pixels brilhantes em mais de 50% das imagens do último ano = estruturas fixas
ultimo_ano = s1.filterDate('2025-11-01', '2026-11-01').select('VV')
print('Imagens usadas:', ultimo_ano.size().getInfo())

frequencia = ultimo_ano.map(lambda img: img.gt(-8)).mean().clip(area)
fixos = frequencia.gt(0.5)

navios_filtrados = navios.updateMask(fixos.Not())

Map.addLayer(fixos.And(agua).selfMask(), {'palette': ['FFFF00']}, 'Estruturas fixas (na água)')
Map.addLayer(navios_filtrados, {'palette': ['00FF00']}, 'Navios (filtrados)')
Map

Imagens usadas: 67


Map(center=[-32.12997948235455, -52.04999932811431], controls=(WidgetControl(options=['position', 'transparent…

In [7]:
Map = geemap.Map()
Map.centerObject(area, 12)
Map.addLayer(imagem, {'min': -25, 'max': 0}, f'Sentinel-1 VV ({data})')
Map.addLayer(fixos.And(agua).selfMask(), {'palette': ['FFFF00']}, 'Estruturas fixas (na água)')
Map.addLayer(navios_filtrados, {'palette': ['00FF00']}, 'Navios (filtrados)')
Map

Map(center=[-32.12997948235455, -52.04999932811431], controls=(WidgetControl(options=['position', 'transparent…

In [8]:
# Afasta a máscara de água 100 m da costa
agua_segura = agua.focal_min(radius=100, units='meters')

# Amplia as estruturas fixas em 50 m para cobrir as bordas
fixos_amplos = fixos.focal_max(radius=50, units='meters')

alvos = (navios_filtrados
         .updateMask(agua_segura)
         .updateMask(fixos_amplos.Not())
         .toInt())

# Agrupa pixels vizinhos em objetos e conta quantos pixels cada um tem
vetores = alvos.reduceToVectors(
    geometry=area, scale=10, geometryType='polygon',
    eightConnected=True, labelProperty='classe',
    reducer=ee.Reducer.countEvery(), maxPixels=1e9
)

# Descarta objetos muito pequenos (ruído) e calcula posição e área
def descrever(f):
    centro = f.geometry().centroid(1).coordinates()
    return f.set({
        'longitude': centro.get(0),
        'latitude': centro.get(1),
        'pixels': f.get('count'),
        'area_m2': f.geometry().area(1)
    })

alvos_navios = vetores.filter(ee.Filter.gte('count', 5)).map(descrever)

tabela = geemap.ee_to_df(alvos_navios, columns=['latitude', 'longitude', 'pixels', 'area_m2'])
print('Navios detectados:', len(tabela))

# Mapa novo, só com as camadas atuais
Map = geemap.Map()
Map.centerObject(area, 12)
Map.addLayer(imagem, {'min': -25, 'max': 0}, f'Sentinel-1 VV ({data})')
Map.addLayer(alvos_navios.map(lambda f: ee.Feature(f.geometry().centroid(1))), {'color': 'red'}, 'Navios (centro)')
display(tabela.round(5))
Map



Navios detectados: 27


,latitude,longitude,pixels,area_m2
0,-32.17423,-52.11914,7,700.84429
1,-32.01168,-52.09928,9,901.12427
2,-32.12294,-52.10049,30,3003.67729
3,-32.12728,-52.10053,25,2503.06210
4,-32.12802,-52.10054,43,4305.26616
5,-32.09271,-52.09909,14,1401.72568
6,-32.07818,-52.09084,21,2102.60315
7,-32.08101,-52.08814,59,5907.31790
8,-32.07719,-52.08791,5,500.62059
9,-32.14984,-52.08847,5,500.61301


Map(center=[-32.12997948235455, -52.04999932811431], controls=(WidgetControl(options=['position', 'transparent…

In [9]:
# === Detecção melhorada: filtro de speckle + limiar adaptativo (CFAR simplificado) ===
SENSIBILIDADE = 3      # maior = menos detecções (teste 2.5 a 4)
MIN_PIXELS = 5         # menor que isso = ruído
MAX_PIXELS = 2000      # maior que isso = provavelmente não é navio

img = imagem.select(0)

# Máscaras (iguais às da versão anterior)
agua_segura2 = agua.focal_min(radius=100, units='meters')
fixos_amplos2 = fixos.focal_max(radius=50, units='meters')

# 1. Reduz o speckle
img_suave = img.focal_median(radius=30, units='meters')

# 2. Média e desvio do mar ao redor de cada pixel (janela de 300 m)
mar = img_suave.updateMask(agua_segura2).updateMask(fixos_amplos2.Not())
estat = mar.reduceNeighborhood(
    reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
    kernel=ee.Kernel.circle(300, 'meters')
)
media = estat.select(0)
desvio = estat.select(1)

# 3. Navio = pixel bem mais brilhante que o mar em volta
navios_cfar = (img_suave.gt(media.add(desvio.multiply(SENSIBILIDADE)))
               .updateMask(agua_segura2)
               .updateMask(fixos_amplos2.Not())
               .selfMask()
               .toInt())

# 4. Agrupa em objetos e filtra por tamanho
vetores2 = navios_cfar.reduceToVectors(
    geometry=area, scale=10, geometryType='polygon',
    eightConnected=True, labelProperty='classe',
    reducer=ee.Reducer.countEvery(), maxPixels=1e9, tileScale=4
)

def descrever2(f):
    centro = f.geometry().centroid(1).coordinates()
    return f.set({
        'longitude': centro.get(0),
        'latitude': centro.get(1),
        'pixels': f.get('count'),
        'area_m2': f.geometry().area(1)
    })

alvos_cfar = (vetores2
              .filter(ee.Filter.gte('count', MIN_PIXELS))
              .filter(ee.Filter.lte('count', MAX_PIXELS))
              .map(descrever2))

tabela2 = geemap.ee_to_df(alvos_cfar, columns=['latitude', 'longitude', 'pixels', 'area_m2'])
print('Versão anterior (limiar fixo):', len(tabela))
print('Versão nova (speckle + CFAR):', len(tabela2))

# Mapa comparando as duas versões
Map2 = geemap.Map()
Map2.centerObject(area, 12)
Map2.addLayer(imagem, {'min': -25, 'max': 0}, f'Sentinel-1 ({data})')
Map2.addLayer(alvos_navios.map(lambda f: ee.Feature(f.geometry().centroid(1))), {'color': 'yellow'}, 'Anterior (amarelo)')
Map2.addLayer(alvos_cfar.map(lambda f: ee.Feature(f.geometry().centroid(1))), {'color': 'red'}, 'Nova (vermelho)')
display(tabela2.round(5))
Map2

Versão anterior (limiar fixo): 27
Versão nova (speckle + CFAR): 507


,latitude,longitude,pixels,area_m2
0,-32.06743,-52.22308,10,1001.17447
1,-32.07619,-52.22137,6,600.70416
2,-32.06816,-52.21955,7,700.82332
3,-32.03542,-52.20025,13,1301.55108
4,-32.03293,-52.19898,22,2202.62758
...,...,...,...,...
502,-32.11738,-51.85142,7,700.93366
503,-32.13153,-51.85079,7,700.93176
504,-32.13572,-51.84700,5,500.66582
505,-32.09706,-51.84510,12,1201.60845


Map(center=[-32.12997948235455, -52.04999932811431], controls=(WidgetControl(options=['position', 'transparent…

In [10]:
# === Detecção melhorada v2: speckle + CFAR + piso fixo ===
SENSIBILIDADE = 5      # maior = menos detecções
PISO_DB = -12          # pixel também precisa ser mais brilhante que isso
MIN_PIXELS = 8         # menor que isso = ruído
MAX_PIXELS = 2000      # maior que isso = provavelmente não é navio

img = imagem.select(0)

# Máscaras (iguais às da versão anterior)
agua_segura2 = agua.focal_min(radius=100, units='meters')
fixos_amplos2 = fixos.focal_max(radius=50, units='meters')

# 1. Reduz o speckle
img_suave = img.focal_median(radius=30, units='meters')

# 2. Média e desvio do mar ao redor de cada pixel (janela de 300 m)
mar = img_suave.updateMask(agua_segura2).updateMask(fixos_amplos2.Not())
estat = mar.reduceNeighborhood(
    reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
    kernel=ee.Kernel.circle(300, 'meters')
)
media = estat.select(0)
desvio = estat.select(1)

# 3. Navio = bem mais brilhante que o mar em volta E acima do piso fixo
navios_cfar = (img_suave.gt(media.add(desvio.multiply(SENSIBILIDADE)))
               .And(img_suave.gt(PISO_DB))
               .updateMask(agua_segura2)
               .updateMask(fixos_amplos2.Not())
               .selfMask()
               .toInt())

# 4. Agrupa em objetos e filtra por tamanho
vetores2 = navios_cfar.reduceToVectors(
    geometry=area, scale=10, geometryType='polygon',
    eightConnected=True, labelProperty='classe',
    reducer=ee.Reducer.countEvery(), maxPixels=1e9, tileScale=4
)

def descrever2(f):
    centro = f.geometry().centroid(1).coordinates()
    return f.set({
        'longitude': centro.get(0),
        'latitude': centro.get(1),
        'pixels': f.get('count'),
        'area_m2': f.geometry().area(1)
    })

alvos_cfar = (vetores2
              .filter(ee.Filter.gte('count', MIN_PIXELS))
              .filter(ee.Filter.lte('count', MAX_PIXELS))
              .map(descrever2))

tabela2 = geemap.ee_to_df(alvos_cfar, columns=['latitude', 'longitude', 'pixels', 'area_m2'])
print('Versão anterior (limiar fixo):', len(tabela))
print('Versão nova (speckle + CFAR + piso):', len(tabela2))

# Mapa comparando as duas versões
Map2 = geemap.Map()
Map2.centerObject(area, 12)
Map2.addLayer(imagem, {'min': -25, 'max': 0}, f'Sentinel-1 ({data})')
Map2.addLayer(alvos_navios.map(lambda f: ee.Feature(f.geometry().centroid(1))), {'color': 'yellow'}, 'Anterior (amarelo)')
Map2.addLayer(alvos_cfar.map(lambda f: ee.Feature(f.geometry().centroid(1))), {'color': 'red'}, 'Nova (vermelho)')
display(tabela2.round(5))
Map2

Versão anterior (limiar fixo): 27
Versão nova (speckle + CFAR + piso): 15


,latitude,longitude,pixels,area_m2
0,-32.08127,-52.08818,8,800.99219
1,-32.05205,-52.07189,14,1401.75555
2,-31.99323,-52.06796,39,3904.94491
3,-32.00244,-52.06447,36,3604.56358
4,-31.99345,-52.05956,38,3804.83281
5,-32.03791,-52.04479,11,1101.39650
6,-32.21282,-52.03726,13,1301.60805
7,-32.26375,-52.01461,37,3704.57550
8,-32.12550,-51.97048,13,1301.66974
9,-32.11300,-51.95612,9,901.16376


Map(center=[-32.12997948235455, -52.04999932811431], controls=(WidgetControl(options=['position', 'transparent…

In [11]:
area = ee.Geometry.Point([-52.05, -32.13]).buffer(15000)

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(area)
      .filterDate('2026-01-01', '2026-11-01')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))
      .median()
      .select(['B2', 'B3', 'B4', 'B8', 'B11'])
      .clip(area))

amostras = s2.sample(region=area, scale=20, numPixels=5000)
kmeans = ee.Clusterer.wekaKMeans(6).train(amostras)
classes = s2.cluster(kmeans)

Map = geemap.Map()
Map.centerObject(area, 11)
Map.addLayer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Sentinel-2 (cor real)')
Map.addLayer(classes.randomVisualizer(), {}, 'K-means (6 classes)')
Map

Map(center=[-32.1299846117216, -52.04999949608422], controls=(WidgetControl(options=['position', 'transparent_…

In [12]:
area = ee.Geometry.Point([-52.05, -32.13]).buffer(15000)

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(area)
      .filterDate('2026-01-01', '2026-11-01')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))
      .median()
      .select(['B2', 'B3', 'B4', 'B8', 'B11'])
      .clip(area))

amostras = s2.sample(region=area, scale=20, numPixels=5000)
kmeans = ee.Clusterer.wekaKMeans(10).train(amostras)
classes = s2.cluster(kmeans)

Map = geemap.Map()
Map.centerObject(area, 11)
Map.addLayer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Sentinel-2 (cor real)')
Map.addLayer(classes.randomVisualizer(), {}, 'K-means (10 classes)')
Map

Map(center=[-32.1299846117216, -52.04999949608422], controls=(WidgetControl(options=['position', 'transparent_…

In [13]:
area = ee.Geometry.Point([-52.05, -32.13]).buffer(15000)

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(area)
      .filterDate('2026-01-01', '2026-11-01')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))
      .median()
      .select(['B2', 'B3', 'B4', 'B8', 'B11'])
      .clip(area))

amostras = s2.sample(region=area, scale=20, numPixels=5000)
kmeans = ee.Clusterer.wekaKMeans(15).train(amostras)
classes = s2.cluster(kmeans)

Map = geemap.Map()
Map.centerObject(area, 11)
Map.addLayer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Sentinel-2 (cor real)')
Map.addLayer(classes.randomVisualizer(), {}, 'K-means (15 classes)')
Map

Map(center=[-32.1299846117216, -52.04999949608422], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
area = ee.Geometry.Point([-52.05, -32.13]).buffer(15000)
n_classes = 10

# Imagem única com menos nuvens (sem mediana, sem emendas)
colecao = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(area)
           .filterDate('2026-01-01', '2026-11-01')
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))
           .sort('CLOUDY_PIXEL_PERCENTAGE'))

img = colecao.first()
data = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
print('Imagem de:', data)

s2 = img.select(['B2', 'B3', 'B4', 'B8', 'B11']).clip(area)

# Teste 1: K-means na imagem inteira
amostras = s2.sample(region=area, scale=20, numPixels=5000)
kmeans = ee.Clusterer.wekaKMeans(n_classes).train(amostras)
classes = s2.cluster(kmeans)

# Teste 2: K-means só na terra (água removida pelo MNDWI)
mndwi = s2.normalizedDifference(['B3', 'B11'])
terra = mndwi.lt(0)
s2_terra = s2.updateMask(terra)

amostras_terra = s2_terra.sample(region=area, scale=20, numPixels=5000)
kmeans_terra = ee.Clusterer.wekaKMeans(n_classes).train(amostras_terra)
classes_terra = s2_terra.cluster(kmeans_terra)

Map = geemap.Map()
Map.centerObject(area, 11)
Map.addLayer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, f'Sentinel-2 cor real ({data})')
Map.addLayer(mndwi, {'min': -0.5, 'max': 0.5, 'palette': ['brown', 'white', 'blue']}, 'MNDWI', False)
Map.addLayer(classes.randomVisualizer(), {}, f'K-means imagem inteira ({n_classes} classes)')
Map.addLayer(classes_terra.randomVisualizer(), {}, f'K-means só terra ({n_classes} classes)')
Map

Imagem de: 2026-05-03


Map(center=[-32.1299846117216, -52.04999949608422], controls=(WidgetControl(options=['position', 'transparent_…

: 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage

# Área no mar, perto da barra (mude se os navios estiverem em outro lugar)
xmin, ymin, xmax, ymax = -51.97, -32.17, -51.885, -32.10
regiao = ee.Geometry.Rectangle([xmin, ymin, xmax, ymax])

img = (ee.ImageCollection('COPERNICUS/S1_GRD')
       .filterBounds(regiao)
       .filterDate('2026-10-29', '2026-10-30')
       .filter(ee.Filter.eq('instrumentMode', 'IW'))
       .select('VV')
       .first())

# Baixa a imagem como matriz (em dB) e converte para potência linear
vv_db = geemap.ee_to_numpy(img, region=regiao, scale=20)[:, :, 0].astype(float)
vv = 10 ** (vv_db / 10)
linhas, colunas = vv.shape
print('Tamanho da matriz:', vv.shape)

# CFAR simplificado: compara cada pixel com a média e o desvio do mar ao redor
janela = 41   # tamanho da vizinhança, em pixels
k = 5         # quantos desvios acima do fundo para virar candidato
media = ndimage.uniform_filter(vv, janela)
media2 = ndimage.uniform_filter(vv ** 2, janela)
desvio = np.sqrt(np.maximum(media2 - media ** 2, 0))
candidatos = vv > media + k * desvio

# Agrupa pixels vizinhos em objetos e mede cada um
objetos, n = ndimage.label(candidatos)
print('Candidatos encontrados:', n)

linhas_tabela = []
for i, fatia in enumerate(ndimage.find_objects(objetos), start=1):
    mascara = objetos == i
    valores = vv_db[mascara]
    altura = fatia[0].stop - fatia[0].start
    largura = fatia[1].stop - fatia[1].start
    lin, col = ndimage.center_of_mass(mascara)
    linhas_tabela.append({
        'id': i,
        'area_px': int(mascara.sum()),
        'max_db': valores.max(),
        'media_db': valores.mean(),
        'contraste_db': valores.mean() - 10 * np.log10(media[mascara].mean()),
        'alongamento': max(altura, largura) / min(altura, largura),
        'lon': xmin + (col + 0.5) * (xmax - xmin) / colunas,
        'lat': ymax - (lin + 0.5) * (ymax - ymin) / linhas,
    })

df = pd.DataFrame(linhas_tabela)

# Mostra a imagem com o número de cada candidato
plt.figure(figsize=(10, 10))
plt.imshow(vv_db, cmap='gray', vmin=-25, vmax=0)
for _, c in df.iterrows():
    lin = (ymax - c['lat']) / (ymax - ymin) * linhas
    col = (c['lon'] - xmin) / (xmax - xmin) * colunas
    plt.text(col, lin, str(int(c['id'])), color='red', fontsize=10)
plt.title('Candidatos do CFAR (números em vermelho)')
plt.show()

df.round(3)

Tamanho da matriz: (393, 405)
Candidatos encontrados: 111


In [ ]:
from sklearn.ensemble import RandomForestClassifier

navios = [1, 2]   # TROQUE pelos ids que você acha que são navios

df['navio'] = df['id'].isin(navios).astype(int)

if df['navio'].nunique() < 2:
    print('Marque pelo menos um navio e deixe pelo menos um candidato como não navio.')
else:
    medidas = ['area_px', 'max_db', 'media_db', 'contraste_db', 'alongamento']
    rf = RandomForestClassifier(n_estimators=200, random_state=0)
    rf.fit(df[medidas], df['navio'])

    df['prob_navio'] = rf.predict_proba(df[medidas])[:, 1]

    print('O que o modelo mais usou para decidir:')
    for nome, peso in sorted(zip(medidas, rf.feature_importances_), key=lambda x: -x[1]):
        print(f'  {nome}: {peso:.2f}')

    display(df.sort_values('prob_navio', ascending=False).round(3))